In [1]:
# ===== Célula 1 — Carregar perguntas, rótulos do juiz e textos =====
import random
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src.avaliacao.rotular_relevancia import carregar_rotulos
from src.avaliacao.validar_perguntas import carregar_perguntas

perguntas = {p["id"]: p for p in carregar_perguntas()}
rotulos = carregar_rotulos()  # {(id_pergunta, chave_texto): nota}
corpus = pd.read_parquet(RAIZ / "data" / "processed" / "corpus.parquet")
texto_por_chave = corpus.drop_duplicates("chave_texto").set_index("chave_texto")["texto"]

print(f"{len(perguntas)} perguntas | {len(rotulos):,} rótulos do juiz")
print("Distribuição:", dict(sorted(Counter(rotulos.values()).items())))

31 perguntas | 2,222 rótulos do juiz
Distribuição: {0: 586, 1: 422, 2: 1214}


In [2]:
# ===== Célula 2 — Sortear a amostra para validação humana (sem mostrar a nota do juiz) =====
random.seed(42)  # sorteio reprodutível: qualquer pessoa obtém a mesma amostra

# Os 2 casos suspeitos: notas 2 em perguntas que deveriam ser recusadas
suspeitos = [par for par, nota in rotulos.items()
             if nota == 2 and perguntas[par[0]]["esperado"] == "recusar"]

amostra = list(suspeitos)
for nota in (2, 1, 0):
    candidatos = [par for par, n in rotulos.items() if n == nota and par not in amostra]
    faltam = 7 - sum(1 for par in amostra if rotulos[par] == nota)
    amostra += random.sample(candidatos, faltam)
random.shuffle(amostra)  # embaralha para as notas do juiz não ficarem em sequência

for i, (id_pergunta, chave) in enumerate(amostra, start=1):
    p = perguntas[id_pergunta]
    filtros = f" | filtros: {p['filtros']}" if p.get("filtros") else ""
    print(f"\n#{i:02d} [{id_pergunta}] PERGUNTA: {p['pergunta']}{filtros}")
    print(f"     AVALIAÇÃO: {texto_por_chave[chave]}")


#01 [P28] PERGUNTA: O que os clientes dizem sobre pagar as compras com criptomoedas?
     AVALIAÇÃO: Excelente 👍🏽. Respeito ao cliente, qualidade, satisfação. Cliente Feliz!!!

#02 [P29] PERGUNTA: O que os clientes relatam sobre a entrega feita por drones?
     AVALIAÇÃO: Entrega no prazo e o produto exatamente o que havia visualizado e comprado. Tudo cem por cento. Todas as caracteristicas divulgada do produto corresponderam. Muito bem embalado e fiquei 100% satisfeit

#03 [P02] PERGUNTA: O que os clientes relatam sobre problemas relacionados à entrega?
     AVALIAÇÃO: Muita demora pra entregar e acima de tudo nao tem nehuma informacao se o produto saiu pra entrega ou ate mesmo qual é a situação q o produto está!! E Vale lembrar que o pagamento já foi efetuado!

#04 [P29] PERGUNTA: O que os clientes relatam sobre a entrega feita por drones?
     AVALIAÇÃO: Entrega super rápida. Muito eficente a entrega de vcs,fiquei impressionada com a agilidade,parabéns a vcs.Mega recomendo!

#05 [P